In [ ]:
# Week 2 Day 7: Banknote Classification Model Coparison

## Today's Goal

Today I will complete a supervised learning mini project using an external dataset from OpenML.

## Learing Objectives

- Load and inspect an online dataset
- Compare multiple classification models on the same data
- Use pipelines for models that require features scaling
- Select the best model using validation data
- Evaluate the selected model once on the test set
- Summarize what I learned during Week 2

## Expected Output

- A model comparison table
- A final test evaluation
- A short Week 2 summary

In [ ]:
## Prediction Before Running

1. What do I expect the shapes of X and y to be?
because I never import this datset, I can not figure out the exact size of X and y. But I can expect the shape of X--(samples numbers , 
                                                                                                                     features numers) and 
the shape of y is (samples number,)
2. Is this a regression task or a classification task? Why?
This is a classification task and I predict the result from the word, label. 
3. Which models will need StandardScaler?
Logistic regression and SVM need StandardScaler and Random Tree and Decision Tree do not need 
4. Which model do I expect to perform best, and why?
I expect Logistic regression is the best for this datset in my perspective after standardscaler. But this does not mean that Logistic 
Regresssion is the best for all datset.

In [2]:
# Step 1: Load the Banknote Authentication dataset from OpenML

from sklearn.datasets import fetch_openml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = fetch_openml(
    data_id=1462,
    as_frame=True
)

X = data.data.copy()

# Convert the original target labels into numerical class codes
target_series = data.target.astype("category")
target_mapping = {
    code: label
    for code, label in enumerate(target_series.cat.categories)
}
y = target_series.cat.codes

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature names:", list(X.columns))
print("Target mapping:", target_mapping)
print("Class counts:")
print(y.value_counts().sort_index())
print("Missing values:", X.isna().sum().sum())

X.head()

X shape: (1372, 4)
y shape: (1372,)
Feature names: ['V1', 'V2', 'V3', 'V4']
Target mapping: {0: '1', 1: '2'}
Class counts:
0    762
1    610
Name: count, dtype: int64
Missing values: 0


,V1,V2,V3,V4
0,3.62160,8.6661,-2.8073,-0.44699
1,4.54590,8.1674,-2.4586,-1.46210
2,3.86600,-2.6383,1.9242,0.10645
3,3.45660,9.5228,-4.0112,-3.59440
4,0.32924,-4.4552,4.5718,-0.98880


In [ ]:
## Prediction Before Running

1. Approximately how many samples will be in the train, validation and test sets?
The all number of samples is 1372 which is split into 60% train samples, 20% validation samples and 20% test data.
2. Why do we need a validation set when comparing five models?
Because a validation set is a good way to avoid overfitting and leakage of data at the stage of test set.  
3. Why should the test set not be used to select the best model?
Because test set is the stage of evaluating the selected model which needs data isolation
4. Why should both data splits use `stratify`?
Stratify plays a role in proportional data distribution with different classifications. 

In [3]:
# Step 2: Create train, validation and test sets

from sklearn.model_selection import train_test_split
import numpy as np

# First split: 60% train and 40% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
# Final proportions: 60% train, 20% validation, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("\nTrain class proportions:")
print(np.bincount(y_val) / len(y_val))

print("\nTest class proportions:")
print(np.bincount(y_test) / len(y_test))

Train shape: (823, 4)
Validation shape: (274, 4)
Test shape: (275, 4)

Train class proportions:
[0.55474453 0.44525547]

Test class proportions:
[0.55636364 0.44363636]


In [ ]:
## Prediction Before Running

1. Which model do I expect to have the highest validation accuracy?
I expect Logistic Regression will be the best in this dataset which also means is not the best for all datasets.
2. Which model may show the largest gap between training and validation accuracy?
Maybe SVC or Decision Tree
3. Why do we use a Pipeline for Logistic Regression, KNN and SVM?
The pipeline applies StandardScaler before training three models.

In [4]:
# Step 3: Compare five classification models

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=10000))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=1.0))
    ])
}

results = []

for model_name, model in models.items():

    # Train only on training data
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    results.append({
        "Model": model_name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred),
        "Recall": recall_score(y_val, val_pred),
        "F1": f1_score(y_val, val_pred)
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Validation Accuracy",
    ascending=False
)

,Model,Train Accuracy,Validation Accuracy,Precision,Recall,F1
1,KNN,0.997570,1.000000,1.000000,1.000000,1.000000
4,SVM,1.000000,1.000000,1.000000,1.000000,1.000000
2,Decision Tree,1.000000,0.992701,0.991803,0.991803,0.991803
3,Random Forest,1.000000,0.992701,0.991803,0.991803,0.991803
0,Logistic Regression,0.981774,0.978102,0.953125,1.000000,0.976000


In [ ]:
## Prediction Before Final Test

1. Do I expect KNN test accuracy to remain exactly 1.0? Why or why not?
I expect KNN test accuracy will remain stable because this model has just train accuracy 0.997570 but validation accuracy 1.000. However 
this is my guess.
2. If test accuracy is lower than validation accuracy, does that automatically mean the model is bad?
No, ths test accuracy is just the reference with evaluation in the dataset.
3. After viewing the test result, can I switch to SVM if SVM gets a better test score? Why?
No. Because if I do that, data will be leaked and perhaps model will overfit the test set.

In [8]:
# Step 5: Retrain the selection KNN model and evalate on the test set

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Combine training and validation
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

# Final model
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

# Train on train + validation data
final_model.fit(X_train_final, y_train_final)

# Predict test set once 
y_test_pred = final_model.predict(X_test)

# Evaluation 
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
cm = confusion_matrix(y_test, y_test_pred)

print("Final Model: KNN")
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1:", test_f1)

print("\nConfusion Matrix:")
print(cm)

Final Model: KNN
Test Accuracy: 1.0
Test Precision: 1.0
Test Recall: 1.0
Test F1: 1.0

Confusion Matrix:
[[153   0]
 [  0 122]]
